# 04 — Churn Prediction Model

The prior notebooks told us *who* churns and *when*. This notebook builds a model to predict *which specific customers* are most likely to churn next — enabling targeted, proactive retention outreach.

**The business problem:** Without a predictive model, retention teams either:
- Cast a wide net (expensive: you're calling customers who weren't going to leave anyway), or
- React after churn happens (too late: the revenue is already gone)

A calibrated churn probability score lets the retention team focus their finite time and budget on the customers where intervention is most likely to prevent a cancellation. Even modest improvements in targeting precision translate to significant cost savings and revenue protection.

We'll train and compare two models:
- **Logistic Regression** — fast, interpretable baseline
- **XGBoost** — gradient-boosted trees that typically outperform on tabular data with mixed feature types

## 1. Setup & Load Data

In [ ]:
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../')

from src.preprocessing import get_model_features
from src.model import (
    split,
    train_logistic,
    train_xgboost,
    evaluate,
    cross_validate,
    save_model,
    plot_roc_curves,
    plot_confusion_matrix,
    plot_feature_importance
)

df = pd.read_csv('../data/processed/customers_processed.csv')
print(f'Loaded: {df.shape[0]:,} customers')

## 2. Feature Preparation & Class Balance Check

`get_model_features()` selects the final feature matrix `X` and target vector `y`, handling any remaining encoding and type alignment. We then check the class balance — significant imbalance (e.g. only 15% positive class) affects how we interpret metrics and may require stratified sampling or class weighting.

In [ ]:
X, y = get_model_features(df)

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')
print(f'\nFeatures used ({len(X.columns)}):')
for col in X.columns:
    print(f'  - {col}')

In [ ]:
churn_rate = y.mean()
print(f'Class balance:')
print(f'  Churned (1):  {y.sum():,}  ({churn_rate:.1%})')
print(f'  Retained (0): {(y == 0).sum():,}  ({1 - churn_rate:.1%})')
print()
if churn_rate < 0.2:
    print('Note: Class imbalance detected. XGBoost scale_pos_weight will be set accordingly.')
    print(f'scale_pos_weight = {(1 - churn_rate) / churn_rate:.2f}')
else:
    print('Class balance is reasonable — no special imbalance handling required.')

## 3. Train/Test Split

We use an 80/20 stratified split to ensure both train and test sets have the same churn rate distribution. Stratification is especially important with imbalanced targets where a random split might accidentally put most churners in one partition.

In [ ]:
X_train, X_test, y_train, y_test = split(X, y)

print(f'Train set: {X_train.shape[0]:,} samples  (churn rate: {y_train.mean():.1%})')
print(f'Test set:  {X_test.shape[0]:,} samples   (churn rate: {y_test.mean():.1%})')

## 4. Train Models

### Logistic Regression (Baseline)
Logistic regression is our interpretable baseline. It's fast, its coefficients have a direct probabilistic interpretation, and it establishes a floor that any more complex model should beat.

In [ ]:
print('Training Logistic Regression...')
lr_model = train_logistic(X_train, y_train)
print('Done.')

### XGBoost
XGBoost is the workhorse of tabular ML competitions. It handles non-linear interactions, is robust to outliers, and typically achieves 3–8% higher AUC than logistic regression on real-world churn datasets.

In [ ]:
print('Training XGBoost...')
xgb_model = train_xgboost(X_train, y_train)
print('Done.')

## 5. Evaluation — Model Comparison

We evaluate both models on the held-out test set. For churn prediction, **ROC AUC** is the primary ranking metric (how well the model separates churners from non-churners), while **precision** and **recall** at a chosen threshold determine the operational tradeoffs.

In [ ]:
lr_results = evaluate(lr_model, X_test, y_test)
xgb_results = evaluate(xgb_model, X_test, y_test)

metrics_table = pd.DataFrame({
    'Logistic Regression': lr_results,
    'XGBoost': xgb_results
}).T

print('Model Performance Comparison (test set):')
display(metrics_table.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen'))

## 6. Cross-Validation

A single train/test split can be noisy — if we happened to get an unusually easy or hard test set, our evaluation would be misleading. Cross-validation averages performance across 5 different data splits, giving a much more robust estimate of how the model will generalise to unseen customers.

In [ ]:
print('Running 5-fold cross-validation for XGBoost...')
cv_results = cross_validate(xgb_model, X, y, cv=5)

print('\nCross-Validation Results (XGBoost):')
print(f'  ROC AUC:  {cv_results["roc_auc_mean"]:.4f} ± {cv_results["roc_auc_std"]:.4f}')
print(f'  F1 Score: {cv_results["f1_mean"]:.4f} ± {cv_results["f1_std"]:.4f}')
print(f'  Precision:{cv_results["precision_mean"]:.4f} ± {cv_results["precision_std"]:.4f}')
print(f'  Recall:   {cv_results["recall_mean"]:.4f} ± {cv_results["recall_std"]:.4f}')

## 7. ROC Curves

The ROC curve plots True Positive Rate against False Positive Rate across all possible classification thresholds. A model with no skill produces a diagonal line (AUC = 0.5); a perfect model produces a step function in the top-left corner (AUC = 1.0). We want our curves as far into the top-left as possible.

In [ ]:
# Add y_test to result dicts so plot_roc_curves can compute curves
lr_results['y_test'] = y_test
xgb_results['y_test'] = y_test

fig_roc = plot_roc_curves({'Logistic Regression': lr_results, 'XGBoost': xgb_results})
fig_roc.show()

## 8. Confusion Matrix — XGBoost

The confusion matrix at the default 0.5 threshold shows the breakdown of true positives, false positives, true negatives, and false negatives. In the churn context:
- **False negatives** (predicted 'retain' but actually churned) are the most costly — we miss a customer we could have saved
- **False positives** (predicted 'churn' but actually retained) waste retention budget on unnecessary outreach

In [ ]:
fig_cm = plot_confusion_matrix(xgb_model, X_test, y_test)
fig_cm.show()

## 9. Feature Importance

XGBoost's built-in feature importance quantifies how much each feature contributed to reducing prediction error across all trees. This gives us an initial ranking of predictive drivers — which we'll validate and interpret more rigorously using SHAP in Notebook 05.

In [ ]:
fig_fi = plot_feature_importance(xgb_model, X.columns.tolist())
fig_fi.show()

## 10. Save the Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model_path = save_model(xgb_model, '../models/xgboost_churn.pkl')
print(f'Model saved to: {model_path}')

## 11. Business Framing: Translating Model Output to Retention Strategy

A model accuracy number means little to a Head of Customer Success. The question they care about is: *"How many customers should we call, and how many of those calls will actually prevent a churn?"*

This is determined by the **decision threshold** — the churn probability above which we flag a customer for outreach. The default threshold of 0.5 is rarely optimal for business applications.

In [ ]:
import numpy as np

# Get churn probabilities from XGBoost
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print('Retention targeting analysis at various thresholds:')
print(f'{"Threshold":>12} | {"Flagged":>10} | {"True Churners":>14} | {"Precision":>10} | {"Recall":>8}')
print('-' * 65)

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
for threshold in thresholds:
    predicted_churn = (y_proba >= threshold).astype(int)
    flagged = predicted_churn.sum()
    true_churners_caught = ((predicted_churn == 1) & (y_test == 1)).sum()
    precision = true_churners_caught / flagged if flagged > 0 else 0
    recall = true_churners_caught / y_test.sum() if y_test.sum() > 0 else 0
    print(f'{threshold:>12.1f} | {flagged:>10,} | {true_churners_caught:>14,} | {precision:>10.1%} | {recall:>8.1%}')

print()
print('Key insight: Optimising for business ROI often means choosing a threshold')
print('higher than 0.5. If your retention team can handle 200 outreach calls/month,')
print('set the threshold so that exactly ~200 customers are flagged — then maximise')
print('precision at that volume to ensure the calls are worth the team\'s time.')

## Summary

We've trained and evaluated two churn prediction models:

- **XGBoost outperforms Logistic Regression** on ROC AUC, F1, and precision — the non-linear interactions between tenure, contract type, and charges are better captured by tree-based methods.

- **Cross-validation confirms the results are robust** — the low standard deviation across folds means the model's performance isn't a statistical fluke from a lucky train/test split.

- **Threshold selection is a business decision, not a statistical one.** The optimal threshold depends on the cost of a missed churn vs. the cost of an unnecessary retention call. A 0.4 threshold captures more churners at the cost of lower precision; 0.6 is more surgical but may miss too many at-risk customers.

In Notebook 05, we'll open up the XGBoost 'black box' using SHAP to understand *why* it makes the predictions it does — which is where the real actionable insights live.